In [ ]:
import sys
from datetime import datetime, timedelta
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QLabel
from PyQt5.QtCore import QTimer
from face_detection import FaceDetectorApp
from csv_logger import CSVLogger  # Importiere den Logger

class MainApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Face Detection Launcher")
        self.setGeometry(200, 200, 400, 300)

        # Layout erstellen
        layout = QVBoxLayout()

        # "Start"-Button
        self.start_button = QPushButton("Start Face Detection", self)
        self.start_button.clicked.connect(self.start_face_detection)

        # "Hold"-Button
        self.hold_button = QPushButton("Hold", self)
        self.hold_button.setStyleSheet("background-color: none;")
        self.hold_button.clicked.connect(self.toggle_hold_mode)

        # "End"-Button (NEU)
        self.end_button = QPushButton("End", self)
        self.end_button.setStyleSheet("background-color: none;")  
        self.end_button.clicked.connect(self.close_application)

        # Label für Status-Anzeige
        self.status_label = QLabel("Status: Nicht gestartet", self)

        # Timer-Label
        self.timer_label = QLabel("Timer: 00:00:00", self)

        # Buttons und Labels dem Layout hinzufügen
        layout.addWidget(self.start_button)
        layout.addWidget(self.hold_button)
        layout.addWidget(self.end_button)
        layout.addWidget(self.status_label)
        layout.addWidget(self.timer_label)
        self.setLayout(layout)

        self.face_detector_window = None
        self.start_time = None
        self.mode = 1  
        self.status = 0  

        # Timer zur Aktualisierung der GUI-Zeit
        self.active_time = timedelta(seconds=0)
        self.last_timer_start = None
        self.gui_timer = QTimer()
        self.gui_timer.timeout.connect(self.update_timer_label)
        self.gui_timer.start(1000)

        # CSV-Logger initialisieren
        self.logger = CSVLogger()  

    def close_application(self):
        """Beendet die Anwendung komplett."""
        print("❌ Anwendung wird geschlossen...")
        self.logger.log(self.mode, self.status, self.get_total_time_str())  # Letzter Log-Eintrag
        QApplication.quit()  # Beendet das PyQt5-Fenster
        sys.exit()  # Beendet das ganze Programm

    def start_face_detection(self):
        """Startet die Gesichtserkennung und speichert die Startzeit."""
        if not self.face_detector_window:
            self.start_time = datetime.now()

            log_message = self.get_log_message()
            print(log_message)
            self.logger.log(self.mode, self.status, self.get_total_time_str())

            self.face_detector_window = FaceDetectorApp()
            self.face_detector_window.status_changed.connect(self.update_status)
            self.face_detector_window.show()

    def toggle_hold_mode(self):
        """Schaltet den Hold-Modus an/aus"""
        self.mode = 0 if self.mode == 1 else 1
        self.hold_button.setStyleSheet("background-color: red; color: white;" if self.mode == 0 else "background-color: none;")

        log_message = self.get_log_message()
        print(log_message)
        self.logger.log(self.mode, self.status, self.get_total_time_str())

        self.update_timer_label()

    def update_status(self, status):
        """Aktualisiert den Status"""
        self.status = status
        status_text = "Gesicht erkannt" if status == 1 else "Kein Gesicht erkannt"
        self.status_label.setText(f"Status: {status_text}")

        log_message = self.get_log_message()
        print(log_message)
        self.logger.log(self.mode, self.status, self.get_total_time_str())

        self.update_timer_label()

    def update_timer_label(self):
        """Aktualisiert die Timer-Anzeige in der GUI"""
        self.timer_label.setText(f"Timer: {self.get_total_time_str()}")

    def get_total_time(self):
        """Berechnet die gesamte vergangene Zeit (total_time)"""
        if self.status == 1 and self.mode == 1:  
            if self.last_timer_start is None:
                self.last_timer_start = datetime.now()
            elapsed_time = datetime.now() - self.last_timer_start
            return self.active_time + elapsed_time
        else:
            if self.last_timer_start is not None:
                self.active_time += datetime.now() - self.last_timer_start
                self.last_timer_start = None
            return self.active_time  

    def get_total_time_str(self):
        """Gibt die aktuelle Gesamtzeit als String HH:MM:SS zurück"""
        return str(self.get_total_time()).split('.')[0]

    def get_log_message(self):
        """Erzeugt die Log-Nachricht für print und CSV"""
        return f"Mode= {self.mode} Status= {self.status} Timer= {self.get_total_time_str()} Time= {datetime.now().strftime('%H:%M:%S')}"


# Hauptprogramm
if __name__ == "__main__":
    app = QApplication(sys.argv)
    main_window = MainApp()
    main_window.show()
    sys.exit(app.exec_())


Mode= 1 Status= 0 Timer= 0:00:00 Time= 23:17:01


I0000 00:00:1739917022.985317  111158 gl_context.cc:369] GL version: 2.1 (2.1 ATI-4.8.101), renderer: AMD Radeon Pro 450 OpenGL Engine
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1739917022.990455  111560 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Mode= 1 Status= 1 Timer= 0:00:00 Time= 23:17:03
Mode= 0 Status= 1 Timer= 0:00:04 Time= 23:17:07
Mode= 1 Status= 1 Timer= 0:00:04 Time= 23:17:10
❌ Anwendung wird geschlossen...
